# 6 · Test-set inference & error analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/06_analysis/06_error_analysis.ipynb)

**Pipeline stage 6 of 6.** Run the fine-tuned model over the test set, report
**WER and CER** (the same metrics used in training), and produce the
phoneme-level error analysis: insertions/deletions per phoneme, a substitution
confusion matrix, and word-conditioned error tables.

Official test result: **WER 14.63% · CER 11.75%** (467 utterances, 17,455
reference phonemes).

## Setup + phoneme inventory / tokeniser

In [ ]:
!pip -q install transformers torchaudio evaluate jiwer soundfile librosa pandas matplotlib
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter

PHONEME_MAPPING = {'aː','ɛː','eː','iː','oː','uː','yː','øː','aɪ','aʊ','ɔɪ','ɛɪ','ɔʏ','ɐʊ',
 't͡s','p͡f','t͡ʃ','a','ɛ','ɪ','ɔ','ʊ','ʏ','œ','ə','ɐ','e','o','i','u','y','ø',
 'ʃ','ʒ','ç','χ','x','f','v','s','z','h','p','b','t','d','k','ɡ','g','ʔ','m','n','ŋ',
 'l','ʁ','j','r','w','ɥ'}
_P = sorted(PHONEME_MAPPING, key=len, reverse=True)
WORD_DELIM = "|"
C_DEL, C_INS = "#E8743B", "#1B9E8A"

def parse_to_words(s):
    if not isinstance(s,str) or not s.strip(): return []
    words, cur = [], []
    for tok in s.split():
        if tok == WORD_DELIM:
            if cur: words.append(cur); cur=[]
            continue
        if tok in PHONEME_MAPPING: cur.append(tok)
        else:
            i=0
            while i<len(tok):
                for p in _P:
                    if tok.startswith(p,i): cur.append(p); i+=len(p); break
                else: i+=1
    if cur: words.append(cur)
    return words

def flatten(words):
    flat, owner = [], []
    for w in words:
        ws="".join(w)
        for ph in w: flat.append(ph); owner.append(ws)
    return flat, owner

## Alignment + per-utterance errors + WER/CER scorer

In [ ]:
def align(ref, hyp):
    n,m=len(ref),len(hyp)
    d=np.zeros((n+1,m+1),dtype=np.int32); d[:,0]=np.arange(n+1); d[0,:]=np.arange(m+1)
    for i in range(1,n+1):
        ri=ref[i-1]
        for j in range(1,m+1):
            c=0 if ri==hyp[j-1] else 1
            d[i,j]=min(d[i-1,j]+1,d[i,j-1]+1,d[i-1,j-1]+c)
    ops,i,j=[],n,m
    while i>0 or j>0:
        if i>0 and j>0 and d[i,j]==d[i-1,j-1]+(0 if ref[i-1]==hyp[j-1] else 1):
            ops.append(("match" if ref[i-1]==hyp[j-1] else "sub",i-1,j-1)); i-=1; j-=1
        elif i>0 and d[i,j]==d[i-1,j]+1: ops.append(("del",i-1,-1)); i-=1
        else: ops.append(("ins",-1,j-1)); j-=1
    ops.reverse(); return ops

def errors_for_utterance(audio, ref_str, hyp_str):
    rf,ro=flatten(parse_to_words(ref_str)); hf,ho=flatten(parse_to_words(hyp_str))
    rows=[]
    for k,i,j in align(rf,hf):
        if k=="match": continue
        if k=="sub": rows.append((audio,"sub",rf[i],hf[j],ro[i],ho[j]))
        elif k=="del": rows.append((audio,"del",rf[i],"",ro[i],""))
        else: rows.append((audio,"ins","",hf[j],"",ho[j]))
    return rows, len(rf)

def score_metrics(pairs):
    canon=lambda s: " | ".join(" ".join(w) for w in parse_to_words(s))
    ref=[canon(r) for r,_ in pairs]; hyp=[canon(h) for _,h in pairs]
    S=D=I=N=0
    for r,h in pairs:
        rf,_=flatten(parse_to_words(r)); hf,_=flatten(parse_to_words(h)); N+=len(rf)
        for k,_,_ in align(rf,hf):
            S+=k=="sub"; D+=k=="del"; I+=k=="ins"
    wer=(S+D+I)/max(1,N)
    try:
        import evaluate; cer=evaluate.load("cer").compute(predictions=hyp,references=ref)
    except Exception:                       # offline fallback: char-level edit rate
        cs=cd=ci=cn=0
        for r,h in zip(ref,hyp):
            rc,hc=list(r),list(h); cn+=len(rc)
            for k,_,_ in align(rc,hc): cs+=k=="sub"; cd+=k=="del"; ci+=k=="ins"
        cer=(cs+cd+ci)/max(1,cn)
    print(f"  WER : {wer*100:6.2f} %    CER : {cer*100:6.2f} %    (S={S} D={D} I={I} N={N})")
    return {"wer":wer,"cer":cer,"S":S,"D":D,"I":I,"N":N}

## CTC decoding that keeps the `|` word delimiter

In [ ]:
def ctc_decode(pred_ids, processor):
    pad=processor.tokenizer.pad_token_id
    ids=[int(x) for x in pred_ids]; collapsed=[]; prev=None
    for i in ids:
        if i!=prev: collapsed.append(i)
        prev=i
    toks=processor.tokenizer.convert_ids_to_tokens([i for i in collapsed if i!=pad])
    skip=set(getattr(processor.tokenizer,"all_special_tokens",[]))
    return " ".join(t for t in toks if t and t not in skip)

## Inference over the test set

In [ ]:
def run_inference(test_csv, model_dir, text_col="phonetic", audio_col="audio_path"):
    import torch, soundfile as sf
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
    try: import librosa
    except Exception: librosa=None
    processor=Wav2Vec2Processor.from_pretrained(model_dir)
    model=Wav2Vec2ForCTC.from_pretrained(model_dir)
    dev="cuda" if torch.cuda.is_available() else "cpu"; model.to(dev).eval()
    df=pd.read_csv(test_csv); rows=[]; pairs=[]
    for n,r in enumerate(df.itertuples(index=False),1):
        wav,sr=sf.read(getattr(r,audio_col))
        if wav.ndim>1: wav=wav.mean(axis=1)
        if sr!=16000 and librosa: wav=librosa.resample(wav.astype("float32"),orig_sr=sr,target_sr=16000)
        iv=processor(wav,sampling_rate=16000,return_tensors="pt").input_values.to(dev)
        with torch.no_grad(): pred=torch.argmax(model(iv).logits,dim=-1)[0]
        hyp=ctc_decode(pred,processor); ref=getattr(r,text_col)
        e,_=errors_for_utterance(getattr(r,audio_col),ref,hyp); rows+=e; pairs.append((ref,hyp))
        if n%50==0: print(f"  {n}/{len(df)}")
    print(f"\n{len(df)} utterances scored:"); score_metrics(pairs)
    return pd.DataFrame(rows,columns=["audio_path","error_type","ref_phon","hyp_phon","ref_word","hyp_word"])

# errors_df = run_inference("test_phonological_ipa.csv", "../05_finetune/kolsch_wav2vec2_model")
# To re-plot from a saved errors file instead:
# errors_df = pd.read_csv("errors_with_words.csv", keep_default_na=False)

## Plots — insertions/deletions, substitution confusion

In [ ]:
def plot_ins_del(edf, top_n=20, out="insertions_deletions.png"):
    dels=edf[edf.error_type=="del"].ref_phon.value_counts()
    inss=edf[edf.error_type=="ins"].hyp_phon.value_counts()
    ph=sorted(set(dels.index)|set(inss.index),
        key=lambda p:(-(int(dels.get(p,0))+int(inss.get(p,0))),-int(inss.get(p,0)),-int(dels.get(p,0)),p))[:top_n]
    d=[int(dels.get(p,0)) for p in ph]; s=[int(inss.get(p,0)) for p in ph]
    x=np.arange(len(ph)); w=0.42; fig,ax=plt.subplots(figsize=(15,6.3))
    b1=ax.bar(x-w/2,d,w,label="Deletions",color=C_DEL,edgecolor="white")
    b2=ax.bar(x+w/2,s,w,label="Insertions",color=C_INS,edgecolor="white")
    ax.bar_label(b1,padding=2,fontsize=9); ax.bar_label(b2,padding=2,fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(ph,fontsize=14); ax.set_xlabel("Phoneme"); ax.set_ylabel("Count")
    ax.set_title(f"Insertions & Deletions per phoneme (top {top_n})\ntotal: {int(inss.sum())} insertions, {int(dels.sum())} deletions")
    ax.legend(); ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y",alpha=0.25)
    fig.tight_layout(); fig.savefig(out,dpi=150); plt.show()

def plot_confusion(edf, top_n=20, out="substitution_confusion.png"):
    subs=edf[edf.error_type=="sub"]; total=len(subs); inv=Counter()
    for _,r in subs.iterrows(): inv[r.ref_phon]+=1; inv[r.hyp_phon]+=1
    top=[p for p,_ in inv.most_common(top_n)]; idx={p:k for k,p in enumerate(top)}
    M=np.zeros((len(top),len(top)),int); shown=0
    for _,r in subs.iterrows():
        if r.ref_phon in idx and r.hyp_phon in idx: M[idx[r.ref_phon],idx[r.hyp_phon]]+=1; shown+=1
    fig,ax=plt.subplots(figsize=(13,11.5)); im=ax.imshow(M,cmap="YlOrRd")
    ax.set_xticks(range(len(top))); ax.set_xticklabels(top,fontsize=12)
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top,fontsize=12)
    ax.set_xlabel("Predicted phoneme (hyp)"); ax.set_ylabel("Reference phoneme (ref)")
    pct=round(100*shown/total) if total else 0
    ax.set_title(f"Substitution confusion — top {top_n} phonemes ({shown}/{total} = {pct}% of substitutions)")
    thr=M.max()*0.6 if M.max() else 1
    for i in range(len(top)):
        for j in range(len(top)):
            if M[i,j]: ax.text(j,i,M[i,j],ha="center",va="center",fontsize=10,color="white" if M[i,j]>thr else "black")
    fig.colorbar(im,ax=ax,shrink=0.8,label="count"); fig.tight_layout(); fig.savefig(out,dpi=150); plt.show()

# plot_ins_del(errors_df); plot_confusion(errors_df)

## Output

`errors_df` plus the two figures and (optionally) the word-conditioned CSVs.
This is the analysis behind the report and the paper. The errors concentrate on
reduced vowels (`ə`, `ɐ`), the vocalised `/ʁ/`, and Ripuarian lenition pairs
(`t↔d`, `t→s`) — the very segments that carry dialectal information.